# E-Commerce Analytics — Exploratory Data Analysis
**Project:** Sales Analytics Dashboard  
**Dataset:** 1200 synthetic e-commerce transactions (2023–2024)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='Blues_r')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.titlesize'] = 14

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
df_raw = pd.read_csv('../data/ecommerce_data.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
print('Data Types:')
print(df_raw.dtypes)
print('\nMissing Values:')
print(df_raw.isnull().sum())
print('\nBasic Stats:')
df_raw.describe()

## 2. Data Cleaning & Preprocessing

In [ ]:
df = df_raw.copy()

# Convert date
df['order_date'] = pd.to_datetime(df['order_date'])

# Fill missing quantity with median
df['quantity'] = df['quantity'].fillna(df['quantity'].median())

# Fill missing unit_price with per-category median
df['unit_price'] = df.groupby('product_category')['unit_price'] \
                     .transform(lambda x: x.fillna(x.median()))

# Recalculate total_price
df['total_price'] = df['total_price'].fillna(df['unit_price'] * df['quantity'])

# Standardize categories
df['product_category'] = df['product_category'].str.strip().str.title()

# Feature engineering
df['year']       = df['order_date'].dt.year
df['month']      = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%b')
df['quarter']    = df['order_date'].dt.quarter
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)

print(f'Clean shape: {df.shape}')
print(f'Missing after cleaning: {df.isnull().sum().sum()}')
df.head()

## 3. Key Performance Indicators

In [ ]:
total_revenue   = df['total_price'].sum()
total_orders    = df['order_id'].nunique()
avg_order_value = df['total_price'].mean()
unique_customers = df['customer_id'].nunique()
units_sold      = int(df['quantity'].sum())

print(f'{'KPI':<25} {'Value':>15}')
print('-' * 42)
print(f'{'Total Revenue':<25} {"$"+f"{total_revenue:,.2f}":>15}')
print(f'{'Total Orders':<25} {total_orders:>15,}')
print(f'{'Avg Order Value':<25} {"$"+f"{avg_order_value:,.2f}":>15}')
print(f'{'Unique Customers':<25} {unique_customers:>15,}')
print(f'{'Units Sold':<25} {units_sold:>15,}')

## 4. Sales Trends Over Time

In [ ]:
monthly = df.groupby('year_month')['total_price'].sum().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
monthly.plot(ax=ax, color='#4a90d9', linewidth=2.5, marker='o', markersize=5)
ax.fill_between(range(len(monthly)), monthly.values, alpha=0.15, color='#4a90d9')
ax.set_title('Monthly Revenue Trend (2023–2024)', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Revenue by Category

In [ ]:
cat_rev = df.groupby('product_category')['total_price'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = plt.cm.Blues_r(np.linspace(0.3, 0.8, len(cat_rev)))
cat_rev.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Revenue by Category', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# Pie chart
axes[1].pie(cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%',
            startangle=90, colors=colors)
axes[1].set_title('Revenue Share by Category', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Top Products

In [ ]:
top10 = df.groupby('product_name')['total_price'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
top10.plot(kind='barh', ax=ax, color='#2d6a9f')
ax.set_title('Top 10 Products by Revenue', fontweight='bold')
ax.set_xlabel('Revenue ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.tight_layout()
plt.show()

## 7. Customer Behavior Analysis

In [ ]:
cust = df.groupby('customer_id').agg(
    total_spent=('total_price', 'sum'),
    order_count=('order_id', 'count')
).reset_index()

print('Customer Spend Distribution:')
print(cust['total_spent'].describe())
print(f'\nRepeat customers (>1 order): {(cust.order_count > 1).sum()}')
print(f'One-time buyers            : {(cust.order_count == 1).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Spend distribution
axes[0].hist(cust['total_spent'], bins=40, color='#4a90d9', edgecolor='#1e3a5f')
axes[0].set_title('Customer Total Spend Distribution', fontweight='bold')
axes[0].set_xlabel('Total Spend ($)')
axes[0].set_ylabel('Count')

# Order frequency
freq = cust['order_count'].value_counts().sort_index().head(8)
freq.plot(kind='bar', ax=axes[1], color='#f0c040')
axes[1].set_title('Order Frequency per Customer', fontweight='bold')
axes[1].set_xlabel('Number of Orders')
axes[1].set_ylabel('Customers')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 8. Quarterly Performance

In [ ]:
quarterly = df.groupby(['year', 'quarter']).agg(
    revenue=('total_price', 'sum'),
    orders=('order_id', 'count')
).reset_index()
quarterly['label'] = 'Q' + quarterly['quarter'].astype(str) + ' ' + quarterly['year'].astype(str)

fig, ax1 = plt.subplots(figsize=(10, 4))
x = range(len(quarterly))
bars = ax1.bar(x, quarterly['revenue'], color='#4a90d9', alpha=0.85, label='Revenue', width=0.4)
ax1.set_xticks(x)
ax1.set_xticklabels(quarterly['label'])
ax1.set_ylabel('Revenue ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))

ax2 = ax1.twinx()
ax2.plot(x, quarterly['orders'], color='#f0c040', marker='o', linewidth=2.5, label='Orders')
ax2.set_ylabel('Orders')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('Quarterly Revenue & Orders', fontweight='bold')
plt.tight_layout()
plt.show()

quarterly

## 9. Year-over-Year Comparison

In [ ]:
yoy = df.groupby(['year', 'month'])['total_price'].sum().unstack(level=0)
yoy.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 4))
yoy.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Year-over-Year Monthly Revenue', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.legend(title='Year')
plt.tight_layout()
plt.show()

## 10. Correlation Heatmap

In [ ]:
corr = df[['quantity', 'unit_price', 'total_price', 'month', 'quarter']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            linewidths=.5, linecolor='#1e2d47')
ax.set_title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|---|---|
| Total Revenue | ~$578K |
| Total Orders | 1,200 |
| Avg Order Value | ~$482 |
| Unique Customers | ~379 |
| Top Category | Electronics |
| Top Product | Camera |

**Key findings:**
- Electronics dominates revenue due to high unit prices
- Q4 shows a consistent seasonal uplift in both years
- Most customers are one-time buyers — retention opportunity
- Strong positive correlation between unit_price and total_price